# Synthetic Photometry — documentation examples

Companion notebook to the **Synthetic Photometry** documentation page
(`docs/source/synthetic_photometry.rst`). One section per code snippet.

Requires only the core `stellar-spice` install.

In [1]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")

# Compat: NumPy 2 removed the `np.in1d` alias; some astropy installs still
# reference it at import time. No-op on healthy installations.
import numpy as np
if not hasattr(np, "in1d"):
    np.in1d = np.isin

import jax.numpy as jnp
from spice.models import IcosphereModel
from spice.spectrum import Blackbody, simulate_observed_flux

# A solar-like blackbody star and its observed spectrum, used by the
# passband-luminosity section below.
bb = Blackbody()
model = IcosphereModel.construct(1000, 1., 1., bb.solar_parameters, bb.parameter_names)

wavelengths = jnp.linspace(200., 40000., 20000)
flux = simulate_observed_flux(bb.intensity, model, jnp.log10(wavelengths),
                              disable_doppler_shift=True)[:, 0]

[spice] IcosphereModel constructed in 0.7 s


## Passband Luminosities

In [2]:
from spice.spectrum.filter import JohnsonCousinsU, JohnsonCousinsB, JohnsonCousinsV, Bolometric, GaiaG
from spice.spectrum.spectrum import AB_passband_luminosity, luminosity

# Calculate passband luminosities
filters = [JohnsonCousinsU(), JohnsonCousinsB(), JohnsonCousinsV(), Bolometric(), GaiaG()]
passband_lums = [AB_passband_luminosity(f, wavelengths, flux) for f in filters]

for f, lum in zip(filters, passband_lums):
    print(f"{f.name}: {lum:.3f} mag")

Johnson-Cousins U: 25.317 mag
Johnson-Cousins B: 25.520 mag
Johnson-Cousins V: 24.886 mag
Bolometric: 25.797 mag
Gaia G: 24.811 mag


## Solar Luminosity Calculation

In [3]:
import astropy.units as u

# Calculate theoretical solar luminosity
sigma = (5.67e-8 * u.W / (u.m**2) / (u.K**4)).to(u.erg / (u.cm**2) / (u.s) / (u.K**4))
solar_luminosity = 0.9997011 * jnp.sum(model.areas) * (u.solRad.to(u.cm)**2) * sigma * (5772*u.K)**4

print(f"Theoretical luminosity of the Sun: {solar_luminosity:.3e} erg/s")

Theoretical luminosity of the Sun: 3.830e+33 erg/s


## Blackbody Luminosity Offsets

The documentation snippet sweeps `n_vertices` in `[100, 1000, 5000, 10000]`
over 100,000 wavelength points; here the finest meshes are kept but the
wavelength grid is thinned so the notebook executes in a few minutes on a
laptop CPU. Restore the documentation values for the full-fidelity sweep.

In [4]:
from spice.spectrum import luminosity, absolute_bol_luminosity
from spice.spectrum.filter import JohnsonCousinsB, JohnsonCousinsI, GaiaG, JohnsonCousinsV
from spice.spectrum.spectrum import ST_passband_luminosity

def calculate_blackbody_luminosity(n_vertices):
    bb = Blackbody()
    model = IcosphereModel.construct(n_vertices, 1., 1., bb.solar_parameters, bb.parameter_names)

    wavelengths = jnp.linspace(1., 100000., 20000)  # docs: 100000 points
    flux = simulate_observed_flux(bb.intensity, model, jnp.log10(wavelengths), 10.,
                                  chunk_size=1000, disable_doppler_shift=True)

    solar_luminosity = luminosity(bb.flux, model, wavelengths)

    return {
        'n_vertices': len(model.d_vertices),
        'solar_luminosity': solar_luminosity,
        'absolute_bol_luminosity': absolute_bol_luminosity(solar_luminosity),
        'AB_solar_apparent_mag_B': AB_passband_luminosity(JohnsonCousinsB(), wavelengths, flux[:, 0]),
        'AB_solar_apparent_mag_V': AB_passband_luminosity(JohnsonCousinsV(), wavelengths, flux[:, 0]),
        # Gaia filters are photonic and not supported for ST magnitudes
        'ST_solar_apparent_mag_V': ST_passband_luminosity(JohnsonCousinsV(), wavelengths, flux[:, 0]),
    }

# Calculate for different resolutions (docs: [100, 1000, 5000, 10000])
results = [calculate_blackbody_luminosity(n) for n in [100, 1000, 5000]]
for r in results:
    print(r)

[spice] IcosphereModel constructed in 0.3 s
[spice] IcosphereModel constructed in 0.0 s
[spice] IcosphereModel constructed in 0.4 s
{'n_vertices': 312, 'solar_luminosity': Array(1.2266664e+25, dtype=float32), 'absolute_bol_luminosity': Array(25.975582, dtype=float32), 'AB_solar_apparent_mag_B': Array(25.517511, dtype=float32), 'AB_solar_apparent_mag_V': Array(24.884249, dtype=float32), 'ST_solar_apparent_mag_V': Array(24.887957, dtype=float32)}
{'n_vertices': 1272, 'solar_luminosity': Array(1.2230375e+25, dtype=float32), 'absolute_bol_luminosity': Array(25.9788, dtype=float32), 'AB_solar_apparent_mag_B': Array(25.51972, dtype=float32), 'AB_solar_apparent_mag_V': Array(24.886456, dtype=float32), 'ST_solar_apparent_mag_V': Array(24.890163, dtype=float32)}
{'n_vertices': 5112, 'solar_luminosity': Array(1.2221335e+25, dtype=float32), 'absolute_bol_luminosity': Array(25.979605, dtype=float32), 'AB_solar_apparent_mag_B': Array(25.52027, dtype=float32), 'AB_solar_apparent_mag_V': Array(24.887